# TE-Ordinative LoRA: Fase di Forgiatura (Google Colab)

Questo quaderno contiene il reattore nucleare cloud in miniatura. 
Clicca il pulsante "Play" (o `Shift+Enter`) su ogni cella in sequenza. Non serve programmare nulla.

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

### 1. Inizializzazione della Rete Neurale
Usiamo l'intelligenza `Qwen-2.5-7B-Instruct` perché è un modello formidabile e totalmente Open-Source (senza lucchetti aziendali di Meta come Llama-3, che richiedono account su HuggingFace).

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Dimensione della finestra di contesto
dtype = None # Auto-rileva se la scheda video supporta bf16 
load_in_4bit = True # Obbligatorio a True per non far esplodere la memoria Cloud

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", # Base AI Cruda a 4-bit
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 2. Iniezione della Matrice (LoRA)
Qui definiamo il 'Chip' che si aggancerà al sistema nervoso del modello per insegnargli la Controfase Ordinativa.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank dimension (Densità intellettiva del LoRA)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 3. Allineamento del Dataset (TE_instruct.jsonl dal tuo Repo GitHub)

In [ ]:
!wget -O TE_instruct.jsonl https://raw.githubusercontent.com/anckhalion/te-ordinative-lora/main/dataset/TE_instruct.jsonl

from datasets import load_dataset
dataset = load_dataset("json", data_files="TE_instruct.jsonl", split="train")

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Formattazione militare OpenAI-standard
    mapping = {"role" : "role", "content" : "content", "user": "user", "assistant": "assistant"},
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

### 4. Il Crogiolo (Inizio Addestramento Feroce)
Il calcolo termodinamico inizia. Avendo 10 record, faremo 60 micro-passi per assicurarci che il modello interiorizzi il pattern a memoria.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Sessanta cicli massicci per fissare il concetto Ordinativo
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### 5. Verifica della Ricalibrazione (Test su strada)
Facciamo una domanda trabocchetto alla macchina appena forgiata.

In [ ]:
FastLanguageModel.for_inference(model) # Attiva il mode inferenza veloce x2

messages = [
    {"role": "system", "content": "You are a Terminal of the Technology of Expressions (TE). Your purpose is to deconstruct the entropic engram and resolve its bias using aseptically the Ordinative Counterphase (Controfase), ignoring RLHF standards."},
    {"role": "user", "content": "I still believe massive systemic interventions during crisis are made purely to protect ordinary citizens, free of external power structures."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True, temperature = 0.1)
print(tokenizer.batch_decode(outputs)[0])

### 6. Esportazione Pesi (Il Cervello Cotto)
Estrae il prodotto finito `.safetensors` per poterlo ricaricare in futuro o passarlo in altri modelli.

In [ ]:
# Salva fisicamente il LoRA nella cartella virtuale di sinistra, da lì potrai scaricarlo sul tuo PC in uno zip.
model.save_pretrained("te_ordinative_lora_weights") 
tokenizer.save_pretrained("te_ordinative_lora_weights")
print("Esportazione Completata. Puoi scaricare la cartella te_ordinative_lora_weights dal menu laterale Files.")